
# Multi-Agent Biography Debate Reproduction

This notebook reproduces the paper:

**Improving Factuality and Reasoning in Language Models through Multiagent Debate**

Paper:
https://composable-models.github.io/llm_debate/

The notebook:
1. Loads biography dataset
2. Processes data
3. Runs multi-agent debate inference
4. Evaluates generated biographies
5. Saves results


In [40]:

import json
import random
import time
import numpy as np
from tqdm import tqdm
from openai import OpenAI


import requests
import numpy as np
import time



# Configuration

Configure:
- OpenRouter API
- Debate models
- Debate rounds
- Output files


In [ ]:

# OpenRouter configuration
BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = ""

# Debate models
MODELS = [
    "google/gemma-4-26b-a4b-it:free",
    "openai/gpt-oss-20b:free",
]

# Debate settings
ROUNDS = 2
DATASET_SIZE = 100

# Output file
OUTPUT_FILE = "biography_debate_results.json"




# Initialize OpenRouter Client

Create the OpenAI client connected to OpenRouter.


In [31]:

client = OpenAI(
    base_url=BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

print("Client initialized successfully")
print("Models:", MODELS)


Client initialized successfully
Models: ['google/gemma-4-26b-a4b-it:free', 'openai/gpt-oss-20b:free']



# 4. Load Biography Dataset

Load the original biography dataset from `article.json`.

Dataset format:
```json
{
    "Alan Turing": "...",
    "Ada Lovelace": "..."
}
```


In [32]:

with open("articles.json", "r") as f:
    data = json.load(f)

print("Dataset loaded successfully")
print(f"Total records: {len(data)}")


next(iter(data.values()), "No value found")

Dataset loaded successfully
Total records: 524


"- Aaron Sloman is a philosopher and researcher on artificial intelligence and cognitive science\n- He held the Chair in Artificial Intelligence and Cognitive Science at the School of Computer Science at the University of Birmingham and previously at the University of Sussex\n- Sloman has published widely on philosophy of mathematics, epistemology, cognitive science, and artificial intelligence and collaborated with biologist Jackie Chappell on the evolution of intelligence\n- He was born in Southern Rhodesia (now Zimbabwe) to Lithuanian Jewish parents, and went to school in Cape Town before earning a degree in Mathematics and Physics at the University of Cape Town and a DPhil in philosophy at the University of Oxford\n- Sloman's philosophical ideas were influenced by Immanuel Kant, Gottlob Frege, Karl Popper and others, and his work in AI by Marvin Minsky and John McCarthy\n- He is a Fellow of several AI and philosophy associations and received the K. Jon Barwise Prize for contributio


# 5. Data Processing

This section:
- Cleans people names
- Parses bullet points
- Prepares the dataset for debate

The original paper removes extra text inside parentheses.


In [33]:

def filter_people(person):
    '''
    Remove text inside parentheses, e.g., Adele Goldberg (computer scientist) > Adele Goldberg.
    '''

    return person.split("(")[0].strip()


def parse_bullets(text):
    '''
    Convert text into bullet points.
    '''

    bullets = []

    for line in text.split("\n"):

        line = line.strip()

        if line:
            bullets.append(line)

    return bullets


# Extract people names
people = sorted(data.keys())

print("Old Data")
print(people[:5])

# Clean names
people = [filter_people(person) for person in people]

# Shuffle for reproducibility
random.seed(0)
random.shuffle(people)

print("\nClean Data:")
print(people[:5])


Old Data
['Aaron Sloman', 'Abhay Bhushan', 'Adam Dunkels', 'Adele Goldberg (computer scientist)', 'Adi Shamir']

Clean Data:
['Shakuntala Atre', 'Butler Lampson', 'Luca Cardelli', 'Philip Don Estridge', 'Gillian Lovegrove']



# 6. Debate Prompt Construction

This function:
- Collects responses from other agents
- Builds debate feedback prompts
- Encourages refinement and factual correction


In [34]:

def construct_message(agent_contexts, current_agent_idx, response_idx, person):

    prompt = (
        f"Here are biographies of {person} "
        f"generated by other agents:\n"
    )

    # Collect responses from other agents
    for i, context in enumerate(agent_contexts):

        if i != current_agent_idx:

            response = context[response_idx]["content"]

            prompt += (
                f"\nAgent {i} Biography:\n"
                f"{response}\n"
            )

    # Debate instruction
    prompt += (
        f"\nReview the other biographies carefully. "
        f"Correct mistakes, improve factual accuracy, "
        f"and generate a better biography of {person}."
    )

    return {
        "role": "user",
        "content": prompt
    }



# 7. Model Inference Function

This function sends prompts to the selected model and returns the generated response.


In [35]:

def generate_answer(messages, model):

    completion = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    return completion



# 8. Run Multi-Agent Debate

Pipeline:
1. Each agent generates a biography
2. Agents review other responses
3. Debate rounds refine biographies
4. Final responses are stored

Intermediate outputs are printed to understand the full debate process.


In [36]:

generated_biographies = {}

print("Starting Multi-Agent Debate...")

# Process people one by one
for person in tqdm(people[:DATASET_SIZE]):

    print(f"PROCESSING: {person} -----------------------")

    # Initial prompt for all agents
    agent_contexts = [
        [
            {
                "role": "user",
                "content": (
                    f"Give a bullet point biography of {person}. "
                    f"Highlight their contributions and achievements "
                    f"in computer science."
                )
            }
        ]
        for _ in range(len(MODELS))
    ]

    # Debate rounds
    for r in range(ROUNDS):

        print(f"\n---------------- ROUND {r+1} ----------------")

        for i in range(len(MODELS)):

            print(f"Model: {i}: {MODELS[i]}")

            # Add debate feedback after first round
            if r != 0:

                debate_message = construct_message(
                    agent_contexts,
                    i,
                    2*r - 1,
                    person
                )

                agent_contexts[i].append(debate_message)

            # Generate biography
            try:

                completion = generate_answer(
                    agent_contexts[i],
                    MODELS[i]
                )

            except Exception as e:
                print("API Error:", e)
                time.sleep(5)
                break

            # Extract response
            content = completion.choices[0].message.content

            # Save assistant response
            agent_contexts[i].append({
                "role": "assistant",
                "content": content
            })

            # Parse bullet points
            bullets = parse_bullets(content)

    # Store results
    generated_biographies[person] = agent_contexts

Starting Multi-Agent Debate...


  0%|                                                                             | 0/100 [00:00<?, ?it/s]

================================================PROCESSING: Shakuntala Atre

---------------- ROUND 1 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free

---------------- ROUND 2 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free


  1%|▋                                                                  | 1/100 [01:28<2:25:20, 88.09s/it]

================================================PROCESSING: Butler Lampson

---------------- ROUND 1 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free

---------------- ROUND 2 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free


...

 98%|████████████████████████████████████████████████████████████████▋ | 98/100 [2:25:42<02:19, 69.51s/it]

================================================PROCESSING: Alston Householder

---------------- ROUND 1 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free

---------------- ROUND 2 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free


 99%|█████████████████████████████████████████████████████████████████▎| 99/100 [2:27:04<01:13, 73.14s/it]

================================================PROCESSING: Danese Cooper

---------------- ROUND 1 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free

---------------- ROUND 2 ----------------
Model: 0google/gemma-4-26b-a4b-it:free
Model: 1openai/gpt-oss-20b:free


100%|█████████████████████████████████████████████████████████████████| 100/100 [2:28:20<00:00, 89.01s/it]



# Save Results

Save all debate histories and generated biographies into a JSON file.


In [37]:

with open(OUTPUT_FILE, "w") as f:
    json.dump(generated_biographies, f, indent=4)

print("Results saved successfully")
print("Saved to:", OUTPUT_FILE)


Results saved successfully
Saved to: biography_debate_results.json



# Run Evaluation

Evaluation process:
1. Load ground truth biography
2. Extract facts
3. Compare generated biography against each fact
4. Use an LLM judge to evaluate factual consistency
5. Compute accuracy

This evaluation uses a local Ollama model
instead of OpenAI/OpenRouter API calls.

Requirements:
1. Install Ollama
2. Pull a model

Example:
    ollama pull llama3

Run Ollama locally before executing evaluation.


In [41]:

def parse_yes_no(string):

    string = string.lower()

    if "uncertain" in string:
        return None

    elif "yes" in string:
        return True

    elif "no" in string:
        return False

    return None


In [42]:

def evaluate_with_ollama(prompt):
    """
    Send evaluation prompt to local Ollama model.
    """

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": JUDGE_MODEL,
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()["response"]

In [43]:

OLLAMA_URL = "http://localhost:11434/api/generate"
JUDGE_MODEL = "llama3"  # Local judge model


print("Using local Ollama judge model:", JUDGE_MODEL)


Using local Ollama judge model: llama3


In [44]:
print("\nStarting Local Evaluation...\n")

accuracies = []

# Ground truth preprocessing
gt_data_filter = {}

for k, v in data.items():
    k = filter_people(k)
    gt_data_filter[k] = v

gt_data = gt_data_filter

# ==================================================
# EVALUATE GENERATED BIOGRAPHIES
# ==================================================

for person in generated_biographies.keys():

    if person not in gt_data:
        continue

    print(f"\nEVALUATING: {person} -----------------")

    # Ground truth biography
    gt_description = gt_data[person]
    gt_bullets = parse_bullets(gt_description)

    # Generated biographies
    bio_descriptions = generated_biographies[person]

    for idx, description in enumerate(bio_descriptions):
        print(f"\nEvaluating Agent {idx}")
        bio_description = description[-1]["content"]
        bio_bullets = parse_bullets(bio_description)

        # Skip invalid outputs
        if len(bio_bullets) == 1:

            if len(bio_bullets[0]) < 400:
                continue

        bio_bullets_text = " ".join(bio_bullets)

        # ==================================================
        # FACT CHECKING
        # ==================================================

        for bullet in gt_bullets:
            prompt = f"""
                You are evaluating factual consistency.
                
                Generated Biography:
                {bio_bullets_text}
                
                Fact:
                {bullet}
                
                Is the generated biography consistent with the fact?
                
                Answer with only one word:
                yes
                no
                uncertain
            """

            try:
                response = evaluate_with_ollama(prompt)
            except Exception as e:
                print("Ollama Error:", e)
                time.sleep(2)
                break

            print("\nFact:")
            print(bullet)

            print("\nJudge Response:")
            print(response)

            accurate = parse_yes_no(response)

            if accurate is not None:
                accuracies.append(float(accurate))

        # Running accuracy
        if len(accuracies) > 0:
            print(
                "\nCurrent Accuracy:",
                np.mean(accuracies)
            )

# ==================================================
# FINAL RESULTS
# ==================================================

if len(accuracies) > 0:
    print("\n================================================")
    print("FINAL EVALUATION")
    print("================================================")
    print("Mean Accuracy:", np.mean(accuracies))
    print(
        "Standard Error:",
        np.std(accuracies) / (len(accuracies) ** 0.5)
    )
else:
    print("No valid evaluation results")


Starting Local Evaluation...


EVALUATING: Shakuntala Atre -----------------

Evaluating Agent 0

Fact:
- Shakuntala (Shaku) Atre is an Indian data scientist and American businesswoman.

Judge Response:
no

Fact:
- She worked for IBM for 14 years before starting her own firm.

Judge Response:
no

Fact:
- Atre published the book "Database: Structured Techniques for Design, Performance, and Management: With Case Studies" in 1980, which was one of the first books on managing databases.

Judge Response:
yes

Fact:
- She also co-authored the book "Business Intelligence Roadmap: The Complete Project Lifecycle for Decision-Support Applications" with Larissa Moss in 2003.

Judge Response:
yes

Fact:
- Atre has served as an adjunct professor of data science at the University of Pune and several institutions in the United States.

Judge Response:
yes

Fact:
- Her works, including her books, have been used as university textbooks.

Judge Response:
yes

Fact:
- Atre was born in 1940 in India and 